#Exercise Ninja 1 : Deep Neural Network from Scratch

In [1]:
import numpy as np

# ============================================================
# DONNÉES
# ============================================================
np.random.seed(42)

# Dataset : 10 exemples, 4 features chacun
X = np.random.randn(10, 4)

# Labels : 3 classes → one-hot encoding
y = np.eye(3)[np.random.choice(3, 10)]
print("X shape:", X.shape)  # (10, 4)
print("y shape:", y.shape)  # (10, 3)

# ============================================================
# ÉTAPE 1 — Initialiser les poids et biais
# ============================================================
# Couche 1 : 4 entrées → 5 neurones
W1 = np.random.randn(4, 5) * 0.01
b1 = np.zeros((1, 5))

# Couche 2 : 5 → 4 neurones
W2 = np.random.randn(5, 4) * 0.01
b2 = np.zeros((1, 4))

# Couche sortie : 4 → 3 classes
W3 = np.random.randn(4, 3) * 0.01
b3 = np.zeros((1, 3))

print("W1:", W1.shape, "| W2:", W2.shape, "| W3:", W3.shape)

# ============================================================
# ÉTAPE 2 — Fonctions d'activation
# ============================================================
def relu(z):
    return np.maximum(0, z)  # max(0, x) pour chaque élément

def relu_derivative(z):
    return (z > 0).astype(float)  # 1 si z>0, sinon 0

def softmax(z):
    # On soustrait le max pour éviter les problèmes numériques
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# ============================================================
# ÉTAPE 3 — Forward Propagation
# ============================================================
def forward_propagation(X, W1, b1, W2, b2, W3, b3):
    # Couche 1
    Z1 = np.dot(X, W1) + b1    # Somme pondérée
    A1 = relu(Z1)               # Activation ReLU

    # Couche 2
    Z2 = np.dot(A1, W2) + b2
    A2 = relu(Z2)

    # Couche sortie
    Z3 = np.dot(A2, W3) + b3
    A3 = softmax(Z3)            # Softmax pour les probabilités

    # On garde tout en mémoire pour la backprop
    cache = (Z1, A1, Z2, A2, Z3, A3)
    return A3, cache

# ============================================================
# ÉTAPE 4 — Loss : Cross-Entropy
# ============================================================
def compute_loss(y_true, y_pred):
    n = y_true.shape[0]
    # On ajoute 1e-8 pour éviter log(0)
    loss = -np.sum(y_true * np.log(y_pred + 1e-8)) / n
    return loss

# ============================================================
# ÉTAPE 5 — Backpropagation
# ============================================================
def backpropagation(X, y, cache, W1, W2, W3):
    n = X.shape[0]
    Z1, A1, Z2, A2, Z3, A3 = cache

    # Gradient de la sortie (softmax + cross-entropy)
    dZ3 = A3 - y                          # Erreur à la sortie
    dW3 = np.dot(A2.T, dZ3) / n
    db3 = np.sum(dZ3, axis=0, keepdims=True) / n

    # Gradient couche 2
    dA2 = np.dot(dZ3, W3.T)
    dZ2 = dA2 * relu_derivative(Z2)
    dW2 = np.dot(A1.T, dZ2) / n
    db2 = np.sum(dZ2, axis=0, keepdims=True) / n

    # Gradient couche 1
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = np.dot(X.T, dZ1) / n
    db1 = np.sum(dZ1, axis=0, keepdims=True) / n

    gradients = (dW1, db1, dW2, db2, dW3, db3)
    return gradients

# ============================================================
# ÉTAPE 6 — Mise à jour des poids (Gradient Descent)
# ============================================================
def update_weights(W1, b1, W2, b2, W3, b3, gradients, lr=0.01):
    dW1, db1, dW2, db2, dW3, db3 = gradients

    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2
    W3 -= lr * dW3
    b3 -= lr * db3

    return W1, b1, W2, b2, W3, b3

# ============================================================
# ÉTAPE 7 — Boucle d'entraînement
# ============================================================
learning_rate = 0.01
epochs = 1000

for epoch in range(epochs):
    # Forward
    A3, cache = forward_propagation(X, W1, b1, W2, b2, W3, b3)

    # Loss
    loss = compute_loss(y, A3)

    # Backprop
    gradients = backpropagation(X, y, cache, W1, W2, W3)

    # Mise à jour
    W1, b1, W2, b2, W3, b3 = update_weights(
        W1, b1, W2, b2, W3, b3, gradients, learning_rate
    )

    # Afficher toutes les 100 epochs
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Loss: {loss:.4f}")

# Résultat final
A3_final, _ = forward_propagation(X, W1, b1, W2, b2, W3, b3)
predicted = np.argmax(A3_final, axis=1)
true_labels = np.argmax(y, axis=1)
accuracy = np.mean(predicted == true_labels)
print(f"\nPrécision finale : {accuracy * 100:.1f}%")

X shape: (10, 4)
y shape: (10, 3)
W1: (4, 5) | W2: (5, 4) | W3: (4, 3)
Epoch    0 | Loss: 1.0986
Epoch  100 | Loss: 0.9187
Epoch  200 | Loss: 0.8252
Epoch  300 | Loss: 0.7734
Epoch  400 | Loss: 0.7420
Epoch  500 | Loss: 0.7213
Epoch  600 | Loss: 0.7064
Epoch  700 | Loss: 0.6953
Epoch  800 | Loss: 0.6864
Epoch  900 | Loss: 0.6793

Précision finale : 70.0%


#Exercise Ninja 2 : Backpropagation avec Momentum

In [2]:
import numpy as np

# ============================================================
# DONNÉES
# ============================================================
np.random.seed(0)
X = np.random.randn(100, 2)   # 100 exemples, 2 features
y = (X[:, 0] + X[:, 1] > 0).astype(float).reshape(-1, 1)  # Binaire

# ============================================================
# INITIALISATION
# ============================================================
W1 = np.random.randn(2, 4) * 0.01
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.01
b2 = np.zeros((1, 1))

# Velocity (vitesse) pour le momentum — initialisée à zéro
vW1 = np.zeros_like(W1)
vb1 = np.zeros_like(b1)
vW2 = np.zeros_like(W2)
vb2 = np.zeros_like(b2)

# Hyperparamètres
lr = 0.005
momentum = 0.9

# ============================================================
# FONCTIONS
# ============================================================
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

def compute_loss_binary(y_true, y_pred):
    n = y_true.shape[0]
    return -np.mean(y_true * np.log(y_pred + 1e-8) +
                    (1 - y_true) * np.log(1 - y_pred + 1e-8))

# ============================================================
# ENTRAÎNEMENT AVEC ET SANS MOMENTUM
# ============================================================
def train(use_momentum=True, epochs=500):
    # Reset des poids
    W1 = np.random.randn(2, 4) * 0.01
    b1 = np.zeros((1, 4))
    W2 = np.random.randn(4, 1) * 0.01
    b2 = np.zeros((1, 1))

    vW1 = np.zeros_like(W1)
    vb1 = np.zeros_like(b1)
    vW2 = np.zeros_like(W2)
    vb2 = np.zeros_like(b2)

    losses = []

    for epoch in range(epochs):
        # Forward
        Z1 = np.dot(X, W1) + b1
        A1 = relu(Z1)
        Z2 = np.dot(A1, W2) + b2
        A2 = sigmoid(Z2)

        loss = compute_loss_binary(y, A2)
        losses.append(loss)

        # Backprop
        n = X.shape[0]
        dZ2 = A2 - y
        dW2 = np.dot(A1.T, dZ2) / n
        db2 = np.sum(dZ2, axis=0, keepdims=True) / n

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * relu_deriv(Z1)
        dW1 = np.dot(X.T, dZ1) / n
        db1 = np.sum(dZ1, axis=0, keepdims=True) / n

        # Mise à jour
        if use_momentum:
            # Avec momentum : on accumule la vitesse
            vW1 = momentum * vW1 - lr * dW1
            vb1 = momentum * vb1 - lr * db1
            vW2 = momentum * vW2 - lr * dW2
            vb2 = momentum * vb2 - lr * db2

            W1 += vW1
            b1 += vb1
            W2 += vW2
            b2 += vb2
        else:
            # Sans momentum : mise à jour classique
            W1 -= lr * dW1
            b1 -= lr * db1
            W2 -= lr * dW2
            b2 -= lr * db2

        if epoch % 100 == 0:
            label = "MOMENTUM" if use_momentum else "STANDARD"
            print(f"[{label}] Epoch {epoch:3d} | Loss: {loss:.4f}")

    return losses

print("=== SANS MOMENTUM ===")
losses_standard = train(use_momentum=False)

print("\n=== AVEC MOMENTUM ===")
losses_momentum = train(use_momentum=True)

print(f"\nLoss finale SANS momentum : {losses_standard[-1]:.4f}")
print(f"Loss finale AVEC momentum : {losses_momentum[-1]:.4f}")

=== SANS MOMENTUM ===
[STANDARD] Epoch   0 | Loss: 0.6932
[STANDARD] Epoch 100 | Loss: 0.6930
[STANDARD] Epoch 200 | Loss: 0.6928
[STANDARD] Epoch 300 | Loss: 0.6927
[STANDARD] Epoch 400 | Loss: 0.6926

=== AVEC MOMENTUM ===
[MOMENTUM] Epoch   0 | Loss: 0.6931
[MOMENTUM] Epoch 100 | Loss: 0.6921
[MOMENTUM] Epoch 200 | Loss: 0.6890
[MOMENTUM] Epoch 300 | Loss: 0.6587
[MOMENTUM] Epoch 400 | Loss: 0.4884

Loss finale SANS momentum : 0.6926
Loss finale AVEC momentum : 0.2537


#Exercise Ninja 3 : Comparer les fonctions d'activation sur un CNN

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np

# ============================================================
# DONNÉES
# ============================================================
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Reshape pour CNN : (batch, hauteur, largeur, canaux)
x_train = x_train.reshape(-1, 28, 28, 1) / 255.0
x_test  = x_test.reshape(-1, 28, 28, 1) / 255.0

y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test, 10)

# ============================================================
# FONCTION : construire le CNN avec une activation donnée
# ============================================================
def build_cnn(activation_name):
    """
    activation_name : 'relu', 'leaky_relu', ou 'swish'
    """
    if activation_name == 'leaky_relu':
        activation = tf.keras.layers.LeakyReLU(alpha=0.1)
    elif activation_name == 'swish':
        activation = 'swish'
    else:
        activation = 'relu'

    model = models.Sequential()

    # Conv layer 1
    model.add(layers.Conv2D(32, (3, 3), padding='same', input_shape=(28, 28, 1)))
    if activation_name == 'leaky_relu':
        model.add(layers.LeakyReLU(alpha=0.1))
    else:
        model.add(layers.Activation(activation))
    model.add(layers.MaxPooling2D(2, 2))

    # Conv layer 2
    model.add(layers.Conv2D(64, (3, 3), padding='same'))
    if activation_name == 'leaky_relu':
        model.add(layers.LeakyReLU(alpha=0.1))
    else:
        model.add(layers.Activation(activation))
    model.add(layers.MaxPooling2D(2, 2))

    # Couche dense
    model.add(layers.Flatten())
    model.add(layers.Dense(64, activation='relu'))

    # Sortie
    model.add(layers.Dense(10, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# ============================================================
# ENTRAÎNEMENT ET COMPARAISON
# ============================================================
results = {}

for activation in ['relu', 'leaky_relu', 'swish']:
    print(f"\n{'='*40}")
    print(f"  Entraînement avec : {activation.upper()}")
    print(f"{'='*40}")

    model = build_cnn(activation)
    model.fit(
        x_train, y_train,
        epochs=3,              # 3 epochs pour aller vite
        batch_size=64,
        validation_split=0.1,
        verbose=1
    )

    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    results[activation] = acc
    print(f"Précision test avec {activation}: {acc*100:.2f}%")

# ============================================================
# RÉSUMÉ COMPARATIF
# ============================================================
print("\n" + "="*40)
print("RÉSUMÉ COMPARATIF")
print("="*40)
for name, acc in results.items():
    print(f"{name:12s} → {acc*100:.2f}%")

best = max(results, key=results.get)
print(f"\nMeilleure activation : {best}")


  Entraînement avec : RELU
Epoch 1/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 66s 77ms/step - accuracy: 0.9418 - loss: 0.1956 - val_accuracy: 0.9840 - val_loss: 0.0585
Epoch 2/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 64s 76ms/step - accuracy: 0.9831 - loss: 0.0559 - val_accuracy: 0.9880 - val_loss: 0.0426
Epoch 3/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 82s 76ms/step - accuracy: 0.9874 - loss: 0.0389 - val_accuracy: 0.9862 - val_loss: 0.0460
Précision test avec relu: 98.60%

  Entraînement avec : LEAKY_RELU


/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 67s 78ms/step - accuracy: 0.9445 - loss: 0.1833 - val_accuracy: 0.9807 - val_loss: 0.0676
Epoch 2/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 66s 78ms/step - accuracy: 0.9831 - loss: 0.0537 - val_accuracy: 0.9888 - val_loss: 0.0429
Epoch 3/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 82s 78ms/step - accuracy: 0.9884 - loss: 0.0366 - val_accuracy: 0.9863 - val_loss: 0.0432
Précision test avec leaky_relu: 98.85%

  Entraînement avec : SWISH
Epoch 1/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 90ms/step - accuracy: 0.9418 - loss: 0.2003 - val_accuracy: 0.9815 - val_loss: 0.0669
Epoch 2/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 81s 90ms/step - accuracy: 0.9821 - loss: 0.0577 - val_accuracy: 0.9855 - val_loss: 0.0530
Epoch 3/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 75s 89ms/step - accuracy: 0.9877 - loss: 0.0401 - val_accuracy: 0.9882 - val_loss: 0.0458
Précision test avec swish: 98.68%

RÉSUMÉ COMPARATIF
relu         → 98.60%
leaky_relu   → 98.85%
swish        → 98.68%

Meilleure activation : leaky_relu
